# Cell 1: 타겟 분포 분석 및 롱테일(Long-tail)의 파괴력

## 🔍 Cell 1: 타겟 분포 시각화 - "왜 700분대 피크에 집착했는가?"

#### 분석 배경
- 초기 가설: "대부분의 지연은 0이거나 매우 작을 것이다."
- **실제 데이터**: Zero Ratio는 2.73%에 불과하며, 최대 지연 시간이 **715분**에 달하는 극단적인 **롱테일(Long-tail) 분포**를 보임.

#### RMSE 지표의 특성
- RMSE는 오차의 **제곱**에 비례하여 페널티를 부여합니다.
- 700분 지연 구간을 0분으로 예측할 경우, 단 하나의 샘플에서만 **490,000점**($700^2$)의 오차가 누적됩니다.
- 따라서 하위 90%를 맞추는 것보다, 상위 1%의 '폭발적 지연'을 잡아내는 것이 점수 하락(개선)의 핵심입니다.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 타겟 분포 시각화
plt.figure(figsize=(12, 6))
sns.histplot(train['avg_delay_minutes_next_30m'], bins=100, kde=True, color='royalblue')
plt.title('Distribution of Average Delay Minutes (Target)', fontsize=15)
plt.xlabel('Delay Minutes')
plt.ylabel('Frequency')
plt.axvline(train['avg_delay_minutes_next_30m'].quantile(0.99), color='red', linestyle='--', label='99th Percentile')
plt.legend()
plt.show()

print(f"Target Max: {train['avg_delay_minutes_next_30m'].max():.2f} min")
print(f"Target 99th Percentile: {train['avg_delay_minutes_next_30m'].quantile(0.99):.2f} min")

# Cell 2: 물리 법칙 기반 피처 - 트래픽 강도와 지연

## 🏗️ Cell 2: 물리 법칙 기반 피처 - 트래픽 강도(Traffic Intensity)

#### 피처 생성 논리: 대기행렬이론 (Queueing Theory)
- 단순히 주문량(`order_inflow`)이 많다고 지연이 생기지 않습니다. 시스템이 처리할 수 있는 역량(로봇 수/효율) 대비 얼마나 많은 작업이 들어오는지가 중요합니다.
- **Traffic Intensity ($\rho$)** = $\lambda$ (도착률) / $\mu$ (서비스율)
- 시스템의 이용률($\rho$)이 1.0에 도달하면 대기 행렬은 지수적으로 증가하며, 이것이 시뮬레이션상의 '병목 현상'입니다.

#### 시각화 포인트
- 트래픽 강도가 0.8~1.0 구간을 넘어서는 순간 지연 시간이 수직 상승하는 구간을 확인할 수 있습니다.

In [ ]:
# 트래픽 강도 피처 생성 (Little's Law 기반)
train['arrival_rate'] = train['order_inflow_15m'] / 15
train['service_rate'] = train['robot_active'] / (train['avg_trip_distance'] + 1e-5)
train['traffic_intensity'] = train['arrival_rate'] / (train['service_rate'] + 1e-5)

# 상관관계 시각화
plt.figure(figsize=(10, 6))
plt.scatter(train['traffic_intensity'], train['avg_delay_minutes_next_30m'], alpha=0.3, color='forestgreen')
plt.axvline(1.0, color='red', linestyle='--', label='System Saturation (1.0)')
plt.title('Traffic Intensity vs Delay Minutes', fontsize=15)
plt.xlabel('Traffic Intensity (Arrival / Service)')
plt.ylabel('Delay Minutes')
plt.xlim(0, 2.5) # 가시성을 위해 제한
plt.legend()
plt.show()

# Cell 3: 예측값 Calibration - 공격적 저격의 결과

## 🎯 Cell 3: Calibration의 필요성 - 19.4점 → 17.8점

#### 보정 전략 (Upper-Tail Scaling)
- 모델은 평균적으로 학습하려는 경향이 있어, 700분대 피크를 예측할 때 매우 보수적으로(30~50분) 출력합니다.
- **Calibration**: 예측값의 상위 분포를 분석하여 훈련 데이터의 실제 피크 스케일로 강제 펌핑했습니다.
- 결과적으로 리더보드 점수가 개선되었으며, 이는 1위의 8점대 점수 역시 이러한 '피크 저격'에 달려있음을 시사합니다.

In [ ]:
# 보정 전/후 분포 비교 시뮬레이션
# (실제 submission 생성 로직의 요약본)

plt.figure(figsize=(12, 6))
sns.kdeplot(raw_predictions, label='Raw Model Predictions (Conservative)', shade=True)
sns.kdeplot(final_preds, label='Calibrated Predictions (Aggressive)', shade=True)
plt.title('Comparison of Prediction Distributions', fontsize=15)
plt.xlabel('Predicted Delay Minutes')
plt.ylabel('Density')
plt.axvline(32.68, color='blue', linestyle=':', label='Max before Calibration')
plt.axvline(155.41, color='red', linestyle=':', label='Max after Calibration')
plt.legend()
plt.show()

print(f"LB Score Improvement: 19.47 -> 17.81")